# Environment Smoke Run
Capture machine and ML stack details, then run a tiny import/tensor check. No training or environment-variable dump.

In [ ]:
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'setup.py').exists()),
    None,
)
if project_root is None:
    raise RuntimeError('Run this notebook from within the project checkout.')
sys.path.insert(0, str(project_root / 'src'))

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

report = {
    'os': f'{platform.system()} {platform.release()}',
    'os_version': platform.version(),
    'python': platform.python_version(),
    'python_executable': sys.executable,
    'cpu': platform.processor() or platform.machine(),
    'logical_cpu_count': os.cpu_count(),
    'packages': {name: package_version(name) for name in ('torch', 'torchvision', 'numpy', 'scikit-learn', 'matplotlib')},
    'precision_default': None,
    'cuda_build': None,
    'cuda_available': False,
    'gpu': None,
    'gpu_vram_bytes': None,
    'nvidia_smi': None,
}

try:
    import torch
    report['precision_default'] = str(torch.get_default_dtype())
    report['cuda_build'] = torch.version.cuda
    report['cuda_available'] = torch.cuda.is_available()
    if report['cuda_available']:
        device = torch.cuda.get_device_properties(0)
        report['gpu'] = device.name
        report['gpu_vram_bytes'] = device.total_memory
except ImportError as error:
    report['torch_import_error'] = str(error)

if shutil.which('nvidia-smi'):
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=driver_version,name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5, check=False,
    )
    report['nvidia_smi'] = result.stdout.strip() or result.stderr.strip() or f'exit code {result.returncode}'

print(json.dumps(report, indent=2, sort_keys=True))

In [ ]:
import torch
import dlbench

values = torch.tensor([1.0, 2.0, 3.0])
result = values @ values
assert result.item() == 14.0
print(f'dlbench import: OK ({dlbench.__file__})')
print(f'PyTorch tensor smoke: OK (device={values.device}, result={result.item():g})')